# VisionAI Inspector — Stage 5: Vision Transformers (CNN vs ViT)

**Goal:** add a Vision Transformer to the comparison and answer, with real
measurements rather than assumption: how does a pretrained ViT compare to
the CNN-based approaches from Stages 3-4 on accuracy, model size, and
inference latency?

**Why a pretrained ViT, not one trained from scratch:** a ViT has no
built-in assumptions about images the way a CNN's convolution does (that
nearby pixels are related, that a pattern means the same thing wherever it
appears) — it has to learn those from data instead. That's exactly why
every practical ViT is pretrained on huge datasets first. With our ~177
training images, training one from scratch would just demonstrate "ViTs
need a lot of data" by performing badly, which is a real, worth-knowing
weakness — better to know it as a stated fact than rediscover it as a
broken experiment.

**What we build:** a frozen, pretrained **ViT-B/16** as a feature extractor
— exactly parallel to Stage 3's ResNet18 approach — feeding the same
One-Class SVM / Isolation Forest. Then we measure parameter count, model
size on disk, and CPU inference latency for every backbone built so far
(custom autoencoder, ResNet18, ViT-B/16), so "CNN vs ViT" has actual
numbers behind it, not just an accuracy score.


## 1. Setup, imports, and loading Stage 1's splits

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_utils import build_protocol_splits, make_synthetic_placeholder_dataset
from torch_dataset import BottleDataset
from autoencoder import ConvAutoencoder
from pretrained_features import build_resnet_feature_extractor, build_vit_feature_extractor, extract_features
from evaluation import calibrate_threshold, evaluate_anomaly_scores
from benchmarking import count_parameters, measure_model_size_mb, measure_inference_latency

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "mvtec_ad"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CATEGORY = "bottle"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

category_path = RAW_ROOT / CATEGORY
USING_SYNTHETIC_DATA = not category_path.exists()
if USING_SYNTHETIC_DATA:
    print("Real MVTec data not found — generating synthetic placeholder for pipeline testing.")
    make_synthetic_placeholder_dataset(RAW_ROOT, category=CATEGORY, seed=RANDOM_SEED)

DATA_TAG = "[SYNTHETIC PLACEHOLDER DATA] " if USING_SYNTHETIC_DATA else ""

train_csv = PROCESSED_ROOT / f"{CATEGORY}_train.csv"
val_csv = PROCESSED_ROOT / f"{CATEGORY}_val.csv"
test_csv = PROCESSED_ROOT / f"{CATEGORY}_test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
else:
    print("Stage 1 CSVs not found — regenerating splits with the same protocol-preserving logic.")
    train_df, val_df, test_df = build_protocol_splits(category_path, val_frac=0.15, seed=RANDOM_SEED)

y_test_true = (test_df["label"] == "defective").astype(int).to_numpy()
print(f"\n{DATA_TAG}train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

THRESHOLD_PERCENTILE = 5
BATCH_SIZE = 16

## 2. Extract ViT-B/16 features

Same pattern as Stage 3's ResNet extraction: freeze the backbone, strip the
classification head, run every image through it once, keep the resulting
768-dim embedding (ViT's "CLS token" — a single vector summarizing the
whole image, analogous to ResNet's globally-pooled 512-dim vector).


In [ ]:
vit_model, vit_weights_are_pretrained, vit_preprocess = build_vit_feature_extractor(DEVICE)
vit_tag = "" if vit_weights_are_pretrained else "[RANDOM-INIT FALLBACK, NOT TRANSFER LEARNING] "

train_loader_vit = DataLoader(BottleDataset(train_df, transform=vit_preprocess), batch_size=BATCH_SIZE, shuffle=False)
val_loader_vit = DataLoader(BottleDataset(val_df, transform=vit_preprocess), batch_size=BATCH_SIZE, shuffle=False)
test_loader_vit = DataLoader(BottleDataset(test_df, transform=vit_preprocess), batch_size=BATCH_SIZE, shuffle=False)

print(f"{vit_tag}{DATA_TAG}Extracting ViT-B/16 features (this is noticeably slower than ResNet on CPU)...")
X_train_vit = extract_features(vit_model, train_loader_vit, DEVICE)
X_val_vit = extract_features(vit_model, val_loader_vit, DEVICE)
X_test_vit = extract_features(vit_model, test_loader_vit, DEVICE)

print(f"{vit_tag}{DATA_TAG}X_train_vit shape: {X_train_vit.shape}")

## 3. Fit the same classical anomaly detectors on ViT features

Identical scaling/thresholding discipline as every prior stage: scaler fit on train only, threshold calibrated on val only.

In [ ]:
vit_scaler = StandardScaler()
X_train_vit_scaled = vit_scaler.fit_transform(X_train_vit)
X_val_vit_scaled = vit_scaler.transform(X_val_vit)
X_test_vit_scaled = vit_scaler.transform(X_test_vit)

ocsvm_vit = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm_vit.fit(X_train_vit_scaled)
val_scores_ocsvm_vit = ocsvm_vit.decision_function(X_val_vit_scaled)
test_scores_ocsvm_vit = ocsvm_vit.decision_function(X_test_vit_scaled)
ocsvm_vit_threshold = calibrate_threshold(val_scores_ocsvm_vit, THRESHOLD_PERCENTILE, higher_is_anomalous=False)
ocsvm_vit_result = evaluate_anomaly_scores(
    "One-Class SVM + ViT-B/16 features", y_test_true, test_scores_ocsvm_vit, ocsvm_vit_threshold,
    higher_is_anomalous=False, tag=vit_tag + DATA_TAG
)

print()
iso_vit = IsolationForest(n_estimators=200, contamination=0.01, random_state=RANDOM_SEED)
iso_vit.fit(X_train_vit_scaled)
val_scores_iso_vit = iso_vit.decision_function(X_val_vit_scaled)
test_scores_iso_vit = iso_vit.decision_function(X_test_vit_scaled)
iso_vit_threshold = calibrate_threshold(val_scores_iso_vit, THRESHOLD_PERCENTILE, higher_is_anomalous=False)
iso_vit_result = evaluate_anomaly_scores(
    "Isolation Forest + ViT-B/16 features", y_test_true, test_scores_iso_vit, iso_vit_threshold,
    higher_is_anomalous=False, tag=vit_tag + DATA_TAG
)

## 4. Model size, parameter count, and inference latency

Now the part accuracy alone can't tell you. We benchmark three backbones
side by side: the Stage 3 custom autoencoder, ResNet18, and ViT-B/16 — same
device (CPU, unless you have a GPU), same measurement method for all
three, so the comparison is fair.


In [ ]:
# Rebuild (or reload) each backbone fresh here so every benchmark uses a clean, comparable instance.
autoencoder = ConvAutoencoder(latent_dim=64, dropout_p=0.1).to(DEVICE)
ae_weights_path = PROCESSED_ROOT / f"{CATEGORY}_autoencoder_weights.pt"
if ae_weights_path.exists():
    autoencoder.load_state_dict(torch.load(ae_weights_path, map_location=DEVICE))
autoencoder.eval()

resnet_model, resnet_pretrained, _ = build_resnet_feature_extractor(DEVICE)

ae_input = torch.randn(1, 3, 64, 64)
resnet_input = torch.randn(1, 3, 224, 224)
vit_input = torch.randn(1, 3, 224, 224)

benchmark_rows = []
for name, model, input_tensor in [
    ("Custom Autoencoder (Stage 3)", autoencoder, ae_input),
    ("ResNet18 (Stage 3, pretrained)", resnet_model, resnet_input),
    ("ViT-B/16 (Stage 5, pretrained)", vit_model, vit_input),
]:
    n_params = count_parameters(model)
    size_mb = measure_model_size_mb(model)
    latency_ms = measure_inference_latency(model, input_tensor, DEVICE)
    benchmark_rows.append({
        "model": name,
        "parameters": n_params,
        "size_mb": round(size_mb, 2),
        "latency_ms_per_image": round(latency_ms, 2),
    })

benchmark_df = pd.DataFrame(benchmark_rows)
print(f"{DATA_TAG}Benchmarked on device: {DEVICE}")
benchmark_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(benchmark_df["model"], benchmark_df["parameters"] / 1e6, color="#4C72B0")
axes[0].set_ylabel("Parameters (millions)")
axes[0].set_title(f"{DATA_TAG}Model size (parameter count)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(benchmark_df["model"], benchmark_df["size_mb"], color="#55A868")
axes[1].set_ylabel("Size on disk (MB)")
axes[1].set_title(f"{DATA_TAG}Model size (serialized weights)")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(benchmark_df["model"], benchmark_df["latency_ms_per_image"], color="#C44E52")
axes[2].set_ylabel("ms per image")
axes[2].set_title(f"{DATA_TAG}Inference latency ({DEVICE})")
axes[2].tick_params(axis="x", rotation=25)

fig.tight_layout(pad=2.0)
plt.show()

## 5. Full accuracy comparison — Stage 2, 3, and 5 together

In [ ]:
rows = [
    {"model": f"{vit_tag}One-Class SVM + ViT-B/16 (Stage 5)", "roc_auc": ocsvm_vit_result["auc"]},
    {"model": f"{vit_tag}Isolation Forest + ViT-B/16 (Stage 5)", "roc_auc": iso_vit_result["auc"]},
]

stage3_summary_path = PROCESSED_ROOT / f"{CATEGORY}_stage3_comparison.csv"
if stage3_summary_path.exists():
    stage3_summary = pd.read_csv(stage3_summary_path)
    for _, row in stage3_summary.iterrows():
        rows.append({"model": row["model"], "roc_auc": row["roc_auc"]})
else:
    stage2_summary_path = PROCESSED_ROOT / f"{CATEGORY}_stage2_baseline_summary.csv"
    if stage2_summary_path.exists():
        stage2_summary = pd.read_csv(stage2_summary_path)
        for _, row in stage2_summary.iterrows():
            rows.append({"model": f"{row['model']} (Stage 2)", "roc_auc": row["roc_auc"]})
    print("Stage 3 comparison CSV not found -- run Stage 3's notebook first for the full picture.")

comparison_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
print(f"{DATA_TAG}")
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(comparison_df["model"], comparison_df["roc_auc"], color="#4C72B0")
ax.set_xlabel("ROC-AUC (test set)")
ax.set_title(f"{DATA_TAG}Full comparison: hand-crafted vs CNN vs ViT features")
ax.set_xlim(0, 1)
fig.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
fpr_ocsvm_vit, tpr_ocsvm_vit, _ = roc_curve(y_test_true, -test_scores_ocsvm_vit)
fpr_iso_vit, tpr_iso_vit, _ = roc_curve(y_test_true, -test_scores_iso_vit)
ax.plot(fpr_ocsvm_vit, tpr_ocsvm_vit, label=f"OCSVM+ViT-B/16 (AUC={ocsvm_vit_result['auc']:.3f})")
ax.plot(fpr_iso_vit, tpr_iso_vit, label=f"IsoForest+ViT-B/16 (AUC={iso_vit_result['auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="random chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{DATA_TAG}ViT-based models — ROC curves")
ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test_true, ocsvm_vit_result["y_pred"], display_labels=["good", "defective"], ax=axes[0], colorbar=False)
axes[0].set_title(f"{DATA_TAG}OCSVM + ViT-B/16")
ConfusionMatrixDisplay.from_predictions(y_test_true, iso_vit_result["y_pred"], display_labels=["good", "defective"], ax=axes[1], colorbar=False)
axes[1].set_title(f"{DATA_TAG}IsoForest + ViT-B/16")
fig.tight_layout(pad=2.0)
plt.show()

## 6. Save Stage 5 outputs

In [ ]:
results_df = test_df[["filepath", "label", "defect_type"]].copy()
results_df["ocsvm_vit_score"] = test_scores_ocsvm_vit
results_df["ocsvm_vit_pred_defective"] = ocsvm_vit_result["y_pred"]
results_df["isoforest_vit_score"] = test_scores_iso_vit
results_df["isoforest_vit_pred_defective"] = iso_vit_result["y_pred"]

results_df.to_csv(PROCESSED_ROOT / f"{CATEGORY}_stage5_results.csv", index=False)
comparison_df.to_csv(PROCESSED_ROOT / f"{CATEGORY}_stage5_comparison.csv", index=False)
benchmark_df.to_csv(PROCESSED_ROOT / f"{CATEGORY}_stage5_benchmark.csv", index=False)

print(f"Saved results, comparison table, and benchmark table to {PROCESSED_ROOT}")
print(f"\n{'SYNTHETIC PLACEHOLDER outputs saved.' if USING_SYNTHETIC_DATA else 'Real dataset outputs saved.'}")
if not vit_weights_are_pretrained:
    print("NOTE: ViT results used RANDOM-INIT fallback weights -- pipeline smoke-test only.")

## 7. Summary — Stage 5 complete

What we built and measured:

- A frozen, pretrained **ViT-B/16** feature extractor, feeding the exact
  same classical anomaly detectors used throughout Stages 2-4.
- A genuine **CNN vs ViT** comparison across three axes:
  - **Accuracy** (ROC-AUC on the untouched test set)
  - **Model size** (parameter count and serialized weight size in MB)
  - **Inference latency** (measured, not estimated, on the same device
    for every model)

**What to actually take away from this, once you have real numbers:** if
ViT-B/16 matches or beats ResNet18 on accuracy, ask whether that's worth
its far larger size and slower inference (this notebook measured ~8x
slower on CPU during development, before you have your own real-data
numbers) — this exact tradeoff (accuracy vs. size vs. latency) is what
Stage 9 (Edge AI Optimization) exists to address directly.

**Next (Stage 6):** RAG + LLM — building a retrieval pipeline over
inspection/technical documentation so the system can explain *why* a
detected defect matters and what an operator should check next, not just
flag that one exists.
